In [ ]:
import sys
import os
from pathlib import Path

# Setup path
if __name__ == "__main__":
    root_path = str(Path(os.getcwd()).parents[2])
    sys.path.insert(0, root_path)
    if __package__ is None:
        __package__ = "comfyui-image-scorer.external_modules.step03training.text_data"

print(f"Root path: {root_path}")
print(f"Python path configured")


In [ ]:
from shared.paths import text_data_file, scores_file
from shared.io import load_single_jsonl
from tqdm import tqdm
from typing import List, Dict, Any, Tuple

# Load data
text_data = load_single_jsonl(text_data_file)
scores = load_single_jsonl(scores_file)
print(f"Text data: {len(text_data)} lines")
print(f"Scores: {len(scores)} lines")

# Display sample record
if text_data:
    print(f"\nSample text record keys: {list(text_data[0].keys())}")
    print(f"Sample text record: {text_data[0]}")
if scores:
    print(f"\nSample score: {scores[0]}")


In [ ]:
# Organize data by category for analysis
lora_data: Dict[str, List[Tuple[float, float]]] = {}
positive_prompt: Dict[str, List[Tuple[float, float]]] = {}
negative_prompt: Dict[str, List[Tuple[float, float]]] = {}
discrete_data: Dict[str, Dict[str | int, List[float]]] = {}
continuous_data: Dict[str, List[Tuple[float, float]]] = {}

print("Processing text data...")
# Use min length to avoid index errors
process_len = min(len(text_data), len(scores))
print(
    f"Processing {process_len} records (text_data has {len(text_data)}, scores has {len(scores)})"
)

with tqdm(total=process_len, desc="Processing", unit="record") as pbar:
    for i in range(process_len):
        current_line: Dict[str, Any] = text_data[i]
        current_score: float = scores[i]

        # LoRA analysis: weight vs score
        if "lora" in current_line and "lora_weight" in current_line:
            current_lora_line: List[Tuple[float, float]] = (
                list(lora_data[current_line["lora"]])
                if current_line["lora"] in lora_data
                else []
            )
            current_lora_line.append((current_line["lora_weight"], current_score))
            lora_data[current_line["lora"]] = current_lora_line

        # Positive prompts: weight vs score
        if "positive_prompt" in current_line:
            current_positive = current_line["positive_prompt"]
            for prompt, weight in current_positive:
                if prompt not in positive_prompt:
                    positive_prompt[prompt] = []
                positive_prompt[prompt].append((weight, current_score))

        # Negative prompts: weight vs score
        if "negative_prompt" in current_line:
            current_negative = current_line["negative_prompt"]
            for prompt, weight in current_negative:
                if prompt not in negative_prompt:
                    negative_prompt[prompt] = []
                negative_prompt[prompt].append((weight, current_score))

        # Discrete data (category analysis)
        for key, value in list(current_line.items()):
            if key not in ["lora", "lora_weight", "positive_prompt", "negative_prompt"]:
                if type(value) in [str, int]:
                    if key not in discrete_data:
                        discrete_data[key] = {}
                    current_column = discrete_data[key]
                    if value not in current_column:
                        current_column[value] = []
                    current_column[value].append(current_score)
                elif type(value) is float:
                    if key not in continuous_data:
                        continuous_data[key] = []
                    # Round to 1 decimal to reduce fragmentation in matrix analysis
                    rounded_value = round(value, 1)
                    continuous_data[key].append((rounded_value, current_score))

        pbar.update(1)

print(f"\nData organized:")
print(f"  LoRA models: {len(lora_data)}")
print(f"  Positive prompts: {len(positive_prompt)}")
print(f"  Negative prompts: {len(negative_prompt)}")
print(f"  Discrete categories: {len(discrete_data)}")
print(f"  Continuous parameters: {len(continuous_data)}")


## LoRA Analysis

Analyze how LoRA model weights affect image scores

In [ ]:
# Final summary
import json

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

summary = {
    "total_records": len(text_data),
    "lora_models": len(lora_data),
    "positive_prompts": len(positive_prompt),
    "negative_prompts": len(negative_prompt),
    "discrete_categories": len(discrete_data),
    "continuous_parameters": len(continuous_data),
    "score_statistics": {
        "min": min(scores),
        "max": max(scores),
        "mean": sum(scores) / len(scores) if scores else 0,
        "total": len(scores),
    },
}

print("\nSummary:")
print(json.dumps(summary, indent=2))

# Save report
output_path = Path("text_analysis_summary.json")
with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Report saved to: {output_path}")


## Prompt Analysis (Positive)

Analyze how positive prompts and their weights affect scores

In [ ]:
# Positive prompts analysis - print statistics
print("\nPositive Prompts Statistics:")
print("-" * 60)

if positive_prompt:
    # Calculate mean scores per prompt
    prompt_stats = []
    for prompt, weight_score_list in list(positive_prompt.items())[:20]:
        scores_only = [score for weight, score in weight_score_list]
        mean_score = sum(scores_only) / len(scores_only)
        prompt_stats.append((prompt[:50], mean_score, len(scores_only)))

    # Sort by mean score
    prompt_stats.sort(key=lambda x: x[1], reverse=True)

    for prompt, mean, count in prompt_stats[:10]:
        print(f"  {prompt:50s} | mean: {mean:.2f} | count: {count}")
else:
    print("No positive prompts found in data")


## Prompt Analysis (Negative)

Analyze how negative prompts and their weights affect scores

In [ ]:
# Negative prompts analysis - print statistics
print("\nNegative Prompts Statistics:")
print("-" * 60)

if negative_prompt:
    # Calculate mean scores per prompt
    prompt_stats = []
    for prompt, weight_score_list in list(negative_prompt.items())[:20]:
        scores_only = [score for weight, score in weight_score_list]
        mean_score = sum(scores_only) / len(scores_only)
        prompt_stats.append((prompt[:50], mean_score, len(scores_only)))

    # Sort by mean score (ascending - lower scores are bad)
    prompt_stats.sort(key=lambda x: x[1], reverse=False)

    for prompt, mean, count in prompt_stats[:10]:
        print(f"  {prompt:50s} | mean: {mean:.2f} | count: {count}")
else:
    print("No negative prompts found in data")


## Continuous Parameters

Analyze relationships between continuous parameters and scores

In [ ]:
# Continuous parameters analysis - print statistics
# IMPORTANT: Round continuous values to 1 decimal to reduce fragmentation
print("\nContinuous Parameters Statistics (values rounded to 1 decimal):")
print("-" * 60)

if continuous_data:
    for param_name, value_score_list in list(continuous_data.items())[:10]:
        if value_score_list:
            # Round values to 1 decimal place to group similar values
            rounded_values = [round(val, 1) for val, score in value_score_list]
            scores_only = [score for val, score in value_score_list]

            mean_val = sum(rounded_values) / len(rounded_values)
            mean_score = sum(scores_only) / len(scores_only)

            print(f"  {param_name:30s}:")
            print(
                f"    Value range: {min(rounded_values):.1f} - {max(rounded_values):.1f}"
            )
            print(f"    Score correlation: mean score = {mean_score:.2f}")
else:
    print("No continuous parameters found in data")


## Discrete Categories

Analyze categorical data and how different categories affect scores

In [ ]:
# Discrete categories analysis - print statistics
print("\nDiscrete Categories Analysis:")
print("-" * 60)

if discrete_data:
    for category_name, values_dict in list(discrete_data.items())[:10]:
        if values_dict:
            print(f"\n  {category_name}:")
            # Calculate mean score per category value
            value_stats = []
            for value, scores_list in values_dict.items():
                mean_score = sum(scores_list) / len(scores_list)
                value_stats.append((str(value)[:30], mean_score, len(scores_list)))

            # Sort by mean score
            value_stats.sort(key=lambda x: x[1], reverse=True)

            for value, mean, count in value_stats[:5]:
                print(f"    {value:35s}: mean={mean:.2f} (n={count})")
else:
    print("No discrete categories found in data")


## Summary Statistics

Generate summary statistics for the analyzed data

In [ ]:
import json

# Generate summary report
summary = {
    "total_records": len(text_data),
    "lora_models": len(lora_data),
    "positive_prompts": len(positive_prompt),
    "negative_prompts": len(negative_prompt),
    "discrete_categories": len(discrete_data),
    "continuous_parameters": len(continuous_data),
    "score_statistics": {
        "min": min(scores),
        "max": max(scores),
        "mean": sum(scores) / len(scores),
    },
}

# Save report
output_path = Path("text_analysis_summary.json")
with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nSummary Report:")
print(json.dumps(summary, indent=2))
print(f"\nReport saved to: {output_path}")


## Analysis Complete

Text data analysis successfully completed. The above visualizations show:

1. **LoRA Analysis**: How different LoRA models and weights affect image quality scores
2. **Positive Prompt Analysis**: Which positive prompts correlate with high scores
3. **Negative Prompt Analysis**: Which negative prompts help avoid low scores
4. **Continuous Parameters**: Relationships between numerical parameters and scores
5. **Discrete Categories**: How categorical choices affect image scores

Use these insights to understand what generates high-quality images in your dataset.